# baseline v3

이 베이스라인 코드는 `사전학습 모델 로드`, `배치 학습`, `파인튜닝`, `양자화`, `PEFT` 등이 적용된 버전입니다.

Colab의 GPU 환경에서 개발되었습니다.
- 런타임 - 런타임 유형 변경 - GPU로 변경(T4 GPU 등)



# 환경 준비

개발 환경에 필요한 라이브러리 버전을 고정하고 최신 버전으로 라이브러리를 업데이트합니다.

- 아래 셀 실행
- 실행 완료 후 런타임 - 세션 다시 시작

In [1]:
!pip -q install git+https://github.com/huggingface/transformers accelerate
!pip -q install --index-url https://download.pytorch.org/whl/cu121 torch torchvision torchaudio
!pip -q install "peft>=0.13.2" "bitsandbytes==0.46.1" datasets pillow pandas --upgrade

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import torch
print("Torch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())

Torch version: 2.10.0+cu128
CUDA version: 12.8
cuDNN version: 91002


# 데이터 준비

개발에 필요한 데이터를 준비합니다.

- train.csv, train 폴더
- test.csv, test 폴더
- sample_submission.csv

본 베이스라인은 colab에서 구글 드라이브를 마운트하여 사용합니다.

데이터를 압축 해제하는데 몇 분 정도의 시간이 소요됩니다.

#### 실습 참고 내용

    챕터 2-2 합성 데이터 실습
    - 구글 드라이브 마운트 : drive()

In [3]:
# 구글드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 라이브러리, 데이터, 설정

In [4]:
# 🚀 기존 pip install 셀 아래에 추가로 실행하세요.
!pip -q install albumentations


In [5]:
# 🚀 2. 라이브러리 로드 및 초기 세팅
import os, re, math, random
import pandas as pd
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Dict, List, Any
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup # 🚀 linear 대신 더 정밀한 cosine 스케줄러 사용!
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm.auto import tqdm
import albumentations as A

# 이미지 로드 시 픽셀 제한 해제
Image.MAX_IMAGE_PIXELS = None

# 디바이스 GPU 우선 사용 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
print("🚀 Device:", device)

# ==========================================
# 🎯 A100 전용 하이퍼파라미터 세팅
# ==========================================
# 🚀 3B가 아닙니다! 0.9 달성을 위한 7B 거대 모델 로드
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

# 🚀 시드 영혼까지 고정 (대회 재현성 룰 완벽 대비)
SEED = 42
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(SEED)

# ==========================================
# 🎯 데이터셋 로드
# ==========================================
train_df = pd.read_csv("/content/train.csv")
test_df  = pd.read_csv("/content/test.csv")

# ⚠️ 주의: 테스트용으로 1000개만 뽑는 코드입니다.
# 만약 진짜 최종 제출용으로 풀학습을 돌리실 거라면 아래 코드는 '#'을 쳐서 주석 처리하십시오!
# train_df = train_df.sample(n=1000, random_state=SEED).reset_index(drop=True)

print("🚀 데이터 로드 완료! Train 데이터 개수:", len(train_df))

🚀 Device: cuda
🚀 데이터 로드 완료! Train 데이터 개수: 5073


In [6]:
import random
import os

# 🚀 파이토치와 넘파이의 모든 난수를 42로 고정하여 완벽한 재현성 보장
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(42)

# 모델, Processor

7.5GB 정도의 모델 다운로드가 진행됩니다. 10~20분 정도가 소요됩니다.

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - LoRA 구현 : LoraConfig()

In [7]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, # A100 필수 속도/메모리 최적화
)

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

processor = AutoProcessor.from_pretrained(MODEL_ID, max_pixels=1024*1024)
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto"
)

base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable() # 메모리 절약

# 🚀 타겟 정밀 타격 (r=64 개방)
lora_config = LoraConfig(
    r=64,
    lora_alpha=128,
    lora_dropout=0.1,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

trainable params: 190,357,504 || all params: 8,482,524,160 || trainable%: 2.2441


# 프롬프트 템플릿

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - 프롬프트 템플릿 : convert_to_chatml(), formatting_prompts_func()

In [8]:
# ==========================================
# 🎯 프롬프트 템플릿 (학습/추론 공통 사용)
# ==========================================

# 시스템 프롬프트: Qwen 모델이 가장 말을 잘 듣는 강력한 영어 명령어로 통제
SYSTEM_INSTRUCT = (
    "You are a highly accurate visual question answering assistant specializing in recycling and waste classification. "
    "Analyze the image carefully and answer the multiple-choice question. "
    "You MUST answer with exactly one lowercase letter: a, b, c, or d. "
    "Do NOT output any extra text, explanation, or punctuation."
)

# 유저 프롬프트: 군더더기 없이 모델이 직관적으로 읽기 편하게 구조화
def build_mc_prompt(question, a, b, c, d):
    return (
        f"질문: {question}\n\n"
        f"a. {a}\n"
        f"b. {b}\n"
        f"c. {c}\n"
        f"d. {d}\n\n"
        f"정답:"
    )

# Custom Dataset, Collator

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - TensorDataset()

    챕터 5-2 데이터 생성 및 파인튜닝 (향후 학습 분량)
    - IntentDataset()

In [9]:
# 🚀 데이터 증강 파이프라인 (좌우 반전, 약간의 회전, 밝기 조절)
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.3),
    A.RandomBrightnessContrast(p=0.2),
])

class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")

        # 🚀 훈련(Train) 시에만 데이터 증강 적용
        if self.train:
            img_np = np.array(img)
            augmented = train_transform(image=img_np)
            img = Image.fromarray(augmented['image'])

        q = str(row["question"])
        a, b, c, d = str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
        user_text = build_mc_prompt(q, a, b, c, d)

        messages = [
            {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
            {"role":"user","content":[
                {"type":"image","image":img},
                {"type":"text","text":user_text}
            ]}
        ]
        if self.train:
            gold = str(row["answer"]).strip().lower()
            messages.append({"role":"assistant","content":[{"type":"text","text":gold}]})

        return {"messages": messages, "image": img}

# 데이터 콜레이터
class DataCollator:
    def __init__(self, processor, train=True):
        self.processor = processor
        self.train = train

    def __call__(self, batch):
        texts, images = [], []
        for sample in batch:
            text = self.processor.apply_chat_template(
                sample["messages"], tokenize=False, add_generation_prompt=not self.train
            )
            texts.append(text)
            images.append(sample["image"])

        # 🚀 A100의 1024 시력 적용
        enc = self.processor(
            text=texts, images=images, padding=True, return_tensors="pt", max_pixels=1024 * 1024
        )
        if self.train:
            enc["labels"] = enc["input_ids"].clone()
        return enc

# DataLoader

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 데이터로더 정의 : DataLoader()

In [10]:
# 🚀 셔플을 통해 모든 재활용품 클래스가 훈련/검증에 골고루 섞이게 만듭니다.
shuffled_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)

split = int(len(shuffled_df) * 0.9)
train_subset = shuffled_df.iloc[:split]
valid_subset = shuffled_df.iloc[split:]

# --- (이 아래 VQAMCDataset 변환 및 DataLoader 코드는 아까 고친 그대로 유지!) ---
# 🚀 1. VQAMCDataset 형태로 변환 (둘 다 train=True여야 정답이 붙어서 Loss 계산이 가능합니다)
train_ds = VQAMCDataset(train_subset, processor, train=True)
valid_ds = VQAMCDataset(valid_subset, processor, train=True)

# 🚀 2. 데이터로더 (DataCollator 역시 둘 다 True로 세팅해야 라벨(Labels)이 생성됩니다)
train_loader = DataLoader(
    train_ds, batch_size=4, shuffle=True, collate_fn=DataCollator(processor, True)
)
valid_loader = DataLoader(
    valid_ds, batch_size=1, shuffle=False, collate_fn=DataCollator(processor, True) # 여기가 핵심!!
)

# fine-tuning

- 200개만 학습 : 10~20분 소요

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - 모델 정의 : SimpleMLP(), SequentialMLP()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [11]:
from tqdm.auto import tqdm

GRAD_ACCUM = 4
EPOCHS = 3
LR = 1e-4

# 과적합 방지용 weight_decay 0.05
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05)
num_training_steps = EPOCHS * math.ceil(len(train_loader) / GRAD_ACCUM)
scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=int(num_training_steps * 0.1), num_training_steps=num_training_steps
)
scaler = torch.cuda.amp.GradScaler(enabled=True)

best_val_loss = float("inf")
SAVE_DIR = "/content/qwen2_5_vl_7b_lora_best"

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]")

    for step, batch in enumerate(progress_bar, start=1):
        batch = {k: v.to(model.device) for k, v in batch.items()}
        with torch.cuda.amp.autocast(dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        running_loss += loss.item()

        if step % GRAD_ACCUM == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            # 🚀 그래디언트 폭발 방지 (안정성 극대화)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

            progress_bar.set_postfix({"loss": f"{running_loss:.4f}"})
            running_loss = 0.0

    # 검증 루프
    model.eval()
    val_loss = 0.0
    with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):
        for vb in tqdm(valid_loader, desc=f"Epoch {epoch+1} [Valid]"):
            vb = {k: v.to(model.device) for k, v in vb.items()}
            val_loss += model(**vb).loss.item()

    val_loss /= len(valid_loader)
    print(f"📊 [Epoch {epoch+1}] Valid Loss: {val_loss:.4f}")

    # 🚀 [여기에 추가!] 1에폭이 끝날 때마다 VRAM에 쌓인 찌꺼기 강제 청소
    import gc
    gc.collect()
    torch.cuda.empty_cache()

    # 🚀 최고 성능 모델 저장 (Early Stopping)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        model.save_pretrained(SAVE_DIR)
        processor.save_pretrained(SAVE_DIR)
        print(f"🏆 Best Model Saved at Epoch {epoch+1}")


/tmp/ipykernel_19150/2637899253.py:13: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=True)


Epoch 1 [Train]:   0%|          | 0/1142 [00:00<?, ?it/s]

/tmp/ipykernel_19150/2637899253.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16):
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/tmp/ipykernel_19150/2637899253.py:48: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):


Epoch 1 [Valid]:   0%|          | 0/508 [00:00<?, ?it/s]

📊 [Epoch 1] Valid Loss: 6.8034
🏆 Best Model Saved at Epoch 1


Epoch 2 [Train]:   0%|          | 0/1142 [00:00<?, ?it/s]

Epoch 2 [Valid]:   0%|          | 0/508 [00:00<?, ?it/s]

📊 [Epoch 2] Valid Loss: 6.8007
🏆 Best Model Saved at Epoch 2


Epoch 3 [Train]:   0%|          | 0/1142 [00:00<?, ?it/s]

KeyboardInterrupt: 

# inference

30분~1시간 소요

#### 실습 참고 내용

    챕터4-1 RAG 기반 Customer Service AI 에이전트 개발
    - 데이터 파서 : langchain_core.output_parsers(), StrOutputParser()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [13]:
# 가장 학습이 잘 된 베스트 가중치 불러오기 (선택사항, 이미 메모리에 있다면 생략 가능)
# model.load_adapter(SAVE_DIR)

def extract_choice(text: str) -> str:
    match = re.search(r'[a-d]', text.strip().lower())
    return match.group() if match else "c"

model.eval()
preds = []

for i in tqdm(range(len(test_df)), desc="Inference"):
    row = test_df.iloc[i]
    img = Image.open(row["path"]).convert("RGB")
    user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])

    messages = [
        {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
        {"role":"user","content":[{"type":"image","image":img}, {"type":"text","text":user_text}]}
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img], padding=True, return_tensors="pt", max_pixels=1024*1024).to(model.device)

    with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):
        out_ids = model.generate(
            **inputs,
            max_new_tokens=2,    # 🚀 딱 1~2토큰만 생성하도록 강제
            do_sample=False,     # 🚀 랜덤성 배제 (Greedy Search)
            eos_token_id=processor.tokenizer.eos_token_id
        )

    input_len = inputs.input_ids.shape[1]
    output_text = processor.decode(out_ids[0][input_len:], skip_special_tokens=True)

    preds.append(extract_choice(output_text))

submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv("submission.csv", index=False)
print("🏆 Saved submission.csv - 마감 전 돌격하십시오!")

Inference:   0%|          | 0/5074 [00:00<?, ?it/s]

/tmp/ipykernel_19150/912802576.py:24: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.bfloat16):
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


🏆 Saved submission.csv - 마감 전 돌격하십시오!


In [ ]:
# 모델 응답 예시
print(output_text)